# Semantic and Hybrid Search: BM25, Dense Retrieval and Rank Fusion

The full code of [Semantic and Hybrid Search: BM25, Dense Retrieval and Rank Fusion](https://residual-stream.lvtan71.workers.dev/posts/semantic-and-hybrid-search/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/semantic-and-hybrid-search) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/semantic-and-hybrid-search.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 6 min end to end on the RTX 3090 machine that saved the outputs below.
The post's run read a copy of the blog taken mid-session, with a draft of Vector Search that isn't in git; this rebuild has 308 English sections where the post had 309, so the last digits can differ.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 5 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" sentence-transformers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "semantic-and-hybrid-search"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/semantic-and-hybrid-search


In [2]:
!tar xzf blog-post37.tar.gz

## `bm25_worked.py`

BM25 for the query "paged KV cache" on the blog's English sections, term by term: document frequency, idf, saturated count and contribution to the best section's score.

In [3]:
!{sys.executable} bm25_worked.py

query "paged KV cache"; best section: "Borrowing an idea from operating systems" (paged-attention-explained), 231 words, average 279
  paged  in  16 of 308 sections: idf 2.93; appears  2 times here; saturated tf 1.51; contributes 4.43


  kv     in  46 of 308 sections: idf 1.89; appears  4 times here; saturated tf 1.88; contributes 3.57
  cache  in  69 of 308 sections: idf 1.49; appears  2 times here; saturated tf 1.51; contributes 2.26
  score 10.26


The post printed:

```text
query "paged KV cache"; best section: "Borrowing an idea from operating systems" (paged-attention-explained), 231 words, average 280
  paged  in  16 of 309 sections: idf 2.93; appears  2 times here; saturated tf 1.51; contributes 4.44
  kv     in  46 of 309 sections: idf 1.90; appears  4 times here; saturated tf 1.89; contributes 3.58
  cache  in  73 of 309 sections: idf 1.44; appears  2 times here; saturated tf 1.51; contributes 2.18
  score 10.20
```

## `retrieval.py`

The benchmark: each section heading is a query for its own section. How much each encoder cuts off; then BM25, two MiniLM encoders (whole sections and best 80-word passage) and reciprocal rank fusion on four tasks (English, Vietnamese, Vietnamese without diacritics, Vietnamese to English), with recall@1, recall@10, MRR, nDCG@10 and bootstrap intervals; and BM25 on a diacritic-stripped index.

In [4]:
!{sys.executable} retrieval.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (287 > 256). Running this sequence through the model will result in indexing errors


English text, MiniLM (English): 70% of sections and 0% of passages cut off
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (324 > 128). Running this sequence through the model will result in indexing errors


English text, MiniLM (multilingual): 94% of sections and 48% of passages cut off


Vietnamese text, MiniLM (English): 89% of sections and 0% of passages cut off


Vietnamese text, MiniLM (multilingual): 96% of sections and 14% of passages cut off


...

  hybrid: BM25 + multilingual passages   recall@1   26%  recall@10   64%  MRR 0.392  nDCG@10 0.442


  hybrid (English) minus BM25: MRR difference -0.040 [-0.081, -0.001]
  hybrid (multilingual) minus BM25: MRR difference +0.015 [-0.026, +0.057]


  hybrid, multilingual weighted 0.25     recall@1   31%  recall@10   64%  MRR 0.428  nDCG@10 0.473



Vietnamese headings typed without diacritics, against a diacritic-stripped index:


  BM25, stripped index                   recall@1   40%  recall@10   76%  MRR 0.524  nDCG@10 0.576


The post printed:

```text
English text, MiniLM (English): 72% of sections and 0% of passages cut off
English text, MiniLM (multilingual): 94% of sections and 48% of passages cut off
Vietnamese text, MiniLM (English): 89% of sections and 0% of passages cut off
Vietnamese text, MiniLM (multilingual): 96% of sections and 14% of passages cut off

English headings -> English sections: 297 queries, 309 sections
  BM25                                   recall@1   35%  recall@10   73%  MRR 0.480  nDCG@10 0.535
  dense, English, whole section          recall@1   31%  recall@10   72%  MRR 0.440  nDCG@10 0.499
  dense, English, best passage           recall@1   32%  recall@10   71%  MRR 0.458  nDCG@10 0.511
  dense, multilingual, whole section     recall@1   27%  recall@10   60%  MRR 0.374  nDCG@10 0.419
  dense, multilingual, best passage      recall@1   26%  recall@10   64%  MRR 0.380  nDCG@10 0.432
  hybrid: BM25 + English passages        recall@1   40%  recall@10   77%  MRR 0.530  nDCG@10 0.582
  hybrid: BM25 + multilingual passages   recall@1   35%  recall@10   75%  MRR 0.494  nDCG@10 0.550
  hybrid (English) minus BM25: MRR difference +0.051 [+0.016, +0.086]
  hybrid (multilingual) minus BM25: MRR difference +0.015 [-0.024, +0.052]
  hybrid, multilingual weighted 0.25     recall@1   38%  recall@10   74%  MRR 0.516  nDCG@10 0.566

Vietnamese headings -> Vietnamese sections: 257 queries, 270 sections
  BM25                                   recall@1   42%  recall@10   78%  MRR 0.539  nDCG@10 0.591
  dense, English, whole section          recall@1   17%  recall@10   44%  MRR 0.262  nDCG@10 0.294
  dense, English, best passage           recall@1   16%  recall@10   48%  MRR 0.256  nDCG@10 0.297
  dense, multilingual, whole section     recall@1   25%  recall@10   54%  MRR 0.352  nDCG@10 0.386
  dense, multilingual, best passage      recall@1   18%  recall@10   56%  MRR 0.308  nDCG@10 0.358
  hybrid: BM25 + English passages        recall@1   28%  recall@10   67%  MRR 0.414  nDCG@10 0.465
  hybrid: BM25 + multilingual passages   recall@1   35%  recall@10   75%  MRR 0.481  nDCG@10 0.540
  hybrid (English) minus BM25: MRR difference -0.125 [-0.168, -0.085]
  hybrid (multilingual) minus BM25: MRR difference -0.058 [-0.102, -0.015]
  hybrid, multilingual weighted 0.25     recall@1   40%  recall@10   77%  MRR 0.537  nDCG@10 0.588

Vietnamese headings typed without diacritics -> Vietnamese sections: 257 queries, 270 sections
  BM25                                   recall@1   26%  recall@10   58%  MRR 0.366  nDCG@10 0.411
  dense, English, whole section          recall@1   18%  recall@10   44%  MRR 0.260  nDCG@10 0.290
  dense, English, best passage           recall@1   16%  recall@10   49%  MRR 0.255  nDCG@10 0.299
  dense, multilingual, whole section     recall@1   14%  recall@10   34%  MRR 0.209  nDCG@10 0.230
  dense, multilingual, best passage      recall@1    9%  recall@10   30%  MRR 0.165  nDCG@10 0.184
  hybrid: BM25 + English passages        recall@1   20%  recall@10   59%  MRR 0.336  nDCG@10 0.388
  hybrid: BM25 + multilingual passages   recall@1   20%  recall@10   54%  MRR 0.314  nDCG@10 0.357
  hybrid (English) minus BM25: MRR difference -0.030 [-0.066, +0.007]
  hybrid (multilingual) minus BM25: MRR difference -0.053 [-0.089, -0.016]
  hybrid, multilingual weighted 0.25     recall@1   26%  recall@10   59%  MRR 0.368  nDCG@10 0.414

Vietnamese headings -> English sections (cross-lingual, headings matched): 245 queries, 309 sections
  BM25                                   recall@1   27%  recall@10   60%  MRR 0.376  nDCG@10 0.426
  dense, English, whole section          recall@1   13%  recall@10   40%  MRR 0.205  nDCG@10 0.238
  dense, English, best passage           recall@1   13%  recall@10   38%  MRR 0.217  nDCG@10 0.243
  dense, multilingual, whole section     recall@1   27%  recall@10   55%  MRR 0.363  nDCG@10 0.396
  dense, multilingual, best passage      recall@1   21%  recall@10   58%  MRR 0.331  nDCG@10 0.382
  hybrid: BM25 + English passages        recall@1   22%  recall@10   58%  MRR 0.339  nDCG@10 0.389
  hybrid: BM25 + multilingual passages   recall@1   26%  recall@10   62%  MRR 0.385  nDCG@10 0.434
  hybrid (English) minus BM25: MRR difference -0.037 [-0.077, +0.002]
  hybrid (multilingual) minus BM25: MRR difference +0.009 [-0.031, +0.049]
  hybrid, multilingual weighted 0.25     recall@1   31%  recall@10   62%  MRR 0.423  nDCG@10 0.465

Vietnamese headings typed without diacritics, against a diacritic-stripped index:
  BM25, stripped index                   recall@1   40%  recall@10   76%  MRR 0.524  nDCG@10 0.576
```